In [1]:
import openai

openai.__version__

'1.66.3'

In [2]:
import time
import base64
from openai import OpenAI

client = OpenAI()

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")
    
def get_token_stats(response):
    token_stats = {
        "prompt_tokens": response.usage.prompt_tokens,
        "completion_tokens": response.usage.completion_tokens,
        "total_tokens": response.usage.total_tokens,
    }
    return token_stats

def ask_gpt(user_query, base64_image, model="gpt-4o-2024-11-20"):
    start_time = time.time()
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": [
                    { "type": "text", "text": user_query },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{base64_image}",
                        },
                    },
                ],
            }
        ],
    )
    latency = time.time() - start_time
    answer = response.choices[0].message.content
    token_stats = get_token_stats(response)
    return answer, token_stats, latency

def ask_gpt_with_structured_output(user_query, base64_image, response_format, model="gpt-4o-2024-11-20"):
    start_time = time.time()
    response = client.beta.chat.completions.parse(
        model=model,
        messages=[
            {
                "role": "user",
                "content": [
                    { "type": "text", "text": user_query },
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}",},},
                ],
            }
        ],
        response_format=response_format
    )
    latency = time.time() - start_time
    answer = response.choices[0].message.parsed
    token_stats = get_token_stats(response)
    return answer, token_stats, latency

# Object Detection

In [3]:
from pydantic import BaseModel
from typing import List

class Detections(BaseModel):
    detected_class: str
    x_min: float
    x_max: float
    y_min: float
    y_max: float

class Detections(BaseModel):
    detections: List[Detections]

user_query = "Detect the food and its location in the image. " \
"Normalize the coordinates between 0 and 1 based on image width and height."

In [4]:
image_path = "images/brunch.jpeg"
base64_image = encode_image(image_path)
answer, token_stats, latency = ask_gpt_with_structured_output(user_query, 
                                                              base64_image, 
                                                              response_format=Detections)

In [5]:
answer.detections

[Detections(detected_class='Eggs Benedict', x_min=0.038577330078125006, x_max=0.46314970703125, y_min=0.3606770833333333, y_max=0.6822916666666666),
 Detections(detected_class='Fruit', x_min=0.42322412109375, x_max=0.5981767578125, y_min=0.4010416666666667, y_max=0.5677083333333334),
 Detections(detected_class='Toast', x_min=0.6812744140625, x_max=0.896240234375, y_min=0.37109375, y_max=0.6354166666666666),
 Detections(detected_class='Coffee Mug', x_min=0.6826171875, x_max=0.767578125, y_min=0.203125, y_max=0.4830729166666667)]

In [6]:
token_stats

{'prompt_tokens': 1273, 'completion_tokens': 208, 'total_tokens': 1481}

In [7]:
latency

8.807827949523926

In [ ]:
from PIL import Image
from PIL import ImageDraw


image = Image.open(image_path).convert("RGB")
draw = ImageDraw.Draw(image)

width, height = image.size

for i, detection in enumerate(answer.detections):
    x1, x2 = detection.x_min * width, detection.x_max * width
    y1, y2 = detection.y_min * height, detection.y_max * height
    draw.rectangle(((x1, y1), (x2, y2)), outline="blue", width=2)
    draw.text((x1, y1), detection.detected_class, fill="red")

image.show()

In [9]:
image_path = "images/nutrition5k-1.png"
base64_image = encode_image(image_path)
answer, token_stats, latency = ask_gpt_with_structured_output(user_query, 
                                                              base64_image, 
                                                              response_format=Detections)

In [10]:
answer.detections

[Detections(detected_class='Potatoes', x_min=0.35, x_max=0.55, y_min=0.2, y_max=0.35),
 Detections(detected_class='Cantaloupe pieces', x_min=0.25, x_max=0.45, y_min=0.4, y_max=0.55),
 Detections(detected_class='Bacon', x_min=0.55, x_max=0.75, y_min=0.4, y_max=0.8)]

In [11]:
token_stats

{'prompt_tokens': 593, 'completion_tokens': 114, 'total_tokens': 707}

In [ ]:
from PIL import Image
from PIL import ImageDraw


image = Image.open(image_path).convert("RGB")
draw = ImageDraw.Draw(image)
width, height = image.size

for i, detection in enumerate(answer.detections):
    x1, x2 = detection.x_min * width, detection.x_max * width
    y1, y2 = detection.y_min * height, detection.y_max * height
    draw.rectangle(((x1, y1), (x2, y2)), outline="blue", width=2)
    draw.text((x1, y1), detection.detected_class)

image.show()

# Visual Question Answering

In [13]:
user_query = """
Detect the foods in the image.
What are macro nutrients of each food?
What are micro nutrients of each food?
What is the calorie count of each food?
What is the serving size of each food?
"""

In [14]:
image_path = "images/nutrition5k-1.png"
base64_image = encode_image(image_path)
answer, token_stats, latency = ask_gpt(user_query, 
                                       base64_image)

In [15]:
print(answer)

### Foods in the Image:
1. Cooked potatoes (diced)
2. Honeydew melon cubes
3. Cooked bacon strips

---

### Macros, Micros, Calories, and Serving Sizes:  
The following information is approximate and may vary based on preparation methods and size.

#### 1. **Cooked Potatoes (Diced)**  
   - **Macronutrients (per 100g):**
     - Carbohydrates: 17g
     - Protein: 2g
     - Fat: 0.1g
   - **Micronutrients (per 100g):**
     - Potassium: 425mg
     - Vitamin C: 11% of daily value
     - Vitamin B6: 15% of daily value
     - Iron: 1% of daily value
   - **Calories (per 100g):** 77 kcal
   - **Serving Size:** Based on the image, roughly 1 cup, about 150g  
     - **Estimated Calories:** 115-120 kcal

---

#### 2. **Honeydew Melon Cubes**  
   - **Macronutrients (per 100g):**
     - Carbohydrates: 9g
     - Protein: 0.5g
     - Fat: 0.1g
   - **Micronutrients (per 100g):**
     - Vitamin C: 30% of daily value
     - Vitamin B6: 5% of daily value
     - Potassium: 230mg
     - Folate: 8% of d

In [16]:
token_stats

{'prompt_tokens': 474, 'completion_tokens': 560, 'total_tokens': 1034}

In [17]:
image_path = "images/brunch.jpeg"
base64_image = encode_image(image_path)
answer, token_stats, latency = ask_gpt(user_query, 
                                       base64_image)

In [18]:
print(answer)

### Foods Detected in the Image:
1. Poached eggs with hollandaise sauce on meat (possibly ham or bacon) and toast (likely Eggs Benedict or a similar dish).
2. Hash browns or breakfast potatoes.
3. Mixed fresh fruit (grapefruit, pineapple, melon).
4. A slice of toast or banana bread topped with powdered sugar.
5. A mug of black coffee.

---

### Macronutrients & Calories (Approximations Per Serving):
#### 1. Poached Eggs with Hollandaise Sauce and Meat
- **Serving Size**: 1 Benedict (2 poached eggs, 1 English muffin, hollandaise sauce, and slices of meat).
- **Calories**: ~400–500 kcal.
- **Macronutrients**:
  - Proteins: 20–25g (eggs, meat).
  - Fats: 25–30g (hollandaise sauce and meat).
  - Carbohydrates: ~10–15g (English muffin).

#### 2. Hash Browns or Breakfast Potatoes
- **Serving Size**: 1 cup (~150g).
- **Calories**: ~200 kcal.
- **Macronutrients**:
  - Carbohydrates: 30–35g (potatoes).
  - Fats: 5–10g.
  - Proteins: 2–4g.

#### 3. Mixed Fresh Fruit
- **Serving Size**: ~1 cup (~

In [19]:
user_query = """
I have some photos for which I want you to provide me with an estimate of 
the nutritional content. Specifically, I want you to give me an estimate of energy, protein,
total carbohydrate, total fat, dietary fibre, total sugar, saturated fat, polyunsaturated fat,
monounsaturated fat, calcium, iron, vitamin D, sodium, potassium, folate, folic acid, and
vitamin C in the total meal. Please also provide a list of the foods in the photo with an
estimate of the physical weight of each food in grams. Please provide your best point
estimate and do not provide a range. When estimating the weight, please provide a weight
estimate for each individual ingredient you identify in the meal rather than the combined
weights of multiple ingredients." \
"""

In [20]:
image_path = "images/nutrition5k-1.png"
base64_image = encode_image(image_path)
answer, token_stats, latency = ask_gpt(user_query, 
                                       base64_image)

In [21]:
print(answer)

Based on the image, here is an analysis of the foods and an estimated nutritional breakdown. Please note this is an approximation and can vary significantly based on preparation methods and portion sizes.

### Foods Identified:
1. **Potatoes (cubed, cooked)**
   - Weight: ~100 grams

2. **Honeydew melon (cubed)**
   - Weight: ~80 grams

3. **Bacon strips (cooked)**
   - Weight: ~50 grams (assume 3 average strips)

---

### Nutritional Estimate for the Total Meal:
#### 1. **Calories (Energy):** ~280 kcal
#### 2. **Protein:** ~10 g
#### 3. **Total Carbohydrate:** ~25 g
#### 4. **Total Fat:** ~16 g
#### 5. **Dietary Fibre:** ~2 g
#### 6. **Total Sugar:** ~8 g
#### 7. **Saturated Fat:** ~5 g
#### 8. **Polyunsaturated Fat:** ~1 g
#### 9. **Monounsaturated Fat:** ~5 g
#### 10. **Calcium:** ~20 mg
#### 11. **Iron:** ~0.8 mg
#### 12. **Vitamin D:** ~0.2 µg
#### 13. **Sodium:** ~600 mg
#### 14. **Potassium:** ~500 mg
#### 15. **Folate:** ~15 µg
#### 16. **Folic Acid:** ~0 µg (not added syntheti

# Document Understanding

In [22]:
user_query = "How much did I pay for the meal? How much tax did I pay? " \
"Which date is that and what is the name of the restaurant?"

image_path = "images/restaurant-bill.jpg"
base64_image = encode_image(image_path)
answer, token_stats, latency = ask_gpt(user_query, 
                                       base64_image)

In [23]:
print(answer)

- **Total paid for the meal:** $85.88  
- **Tax paid:** $5.88  
- **Date:** December 27, 2022  
- **Name of the restaurant:** Nomade Westport Restaurant  


In [24]:
user_query = "What is the price of Ravioli di Carne? What are its ingredients?"

image_path = "images/menu-pasta-1.jpeg"
base64_image = encode_image(image_path)
answer, token_stats, latency = ask_gpt(user_query, 
                                       base64_image)

In [25]:
print(answer)

The price of Ravioli di Carne is **$23.00**. It is described as **beef stuffed pasta in a light bolognese sauce**.


In [26]:
user_query = "What is the price of spaghetti allo scoglio? What are its ingredients?"

image_path = "images/menu-pasta-2.jpeg"
base64_image = encode_image(image_path)
answer, token_stats, latency = ask_gpt(user_query, 
                                       base64_image)

In [27]:
print(answer)

The price of **Spaghetti allo scoglio** is **$28.00**.

Its ingredients are:
- Chopped garlic
- Mussels
- Shrimps
- Scallops
- Clams
- Cherry tomatoes
